# 理解 DeepSeek R1 论文

本节是一次「论文速读」。我们将以通俗易懂的方式解读 DeepSeek R1 论文，拆解其核心概念与关键发现。

DeepSeek R1 是语言模型训练领域的重要里程碑，特别是在通过强化学习开发推理能力方面。该论文提出了 **GRPO（Group Relative Policy Optimization）**，并验证了：**纯强化学习可以在无需监督微调的情况下发展出推理能力**。

> **TIP**: 在 DeepSeek R1 出现之前，所有主流 LLM 都依赖某种程度的监督微调（SFT）。DeepSeek R1-Zero 的出现挑战了这一惯例。

## 「顿悟时刻」：R1-Zero 的突破性发现

DeepSeek R1-Zero 训练过程中最令人惊讶的发现之一，是出现了所谓的「**顿悟时刻（Aha Moment）**」。

这个现象与人类解题时的「豁然开朗」类似，具体表现为：

1. **初次尝试**：模型对问题做出初步解答
2. **自我察觉**：模型识别到潜在的错误或不一致
3. **自我修正**：基于这种察觉调整解题思路
4. **解释说明**：能够解释为什么新方案更好

以拼图为例，理解这个过程：

```
第一步：「这块拼图颜色对，应该放这里」
察觉：「等等，形状不对」
修正：「啊，它应该放那边」
解释：「因为那个位置的颜色和形状都匹配」
```

**最关键的一点**：这种能力是从 RL 训练中**自然涌现**的，并非被显式编程，也不是从训练数据中记忆来的，而是真正意义上的「学习」。

> **TIP**: 你可以在 Hugging Chat 上试用 DeepSeek R1，亲身体验其推理过程。

## 两个核心模型：R1-Zero vs R1

训练过程产生了两个关键模型：

| 特性 | DeepSeek-R1-Zero | DeepSeek-R1 |
|------|-----------------|-------------|
| 训练方式 | 纯强化学习（Pure RL） | 多阶段（SFT + RL） |
| 监督微调 | 无 | 有 |
| 推理能力 | 涌现式（Emergent） | 经过增强的（Enhanced） |
| AIME 2024 成绩 | 71.0% | 79.8% |
| 主要特点 | 推理能力强，但可读性差 | 推理强 + 语言一致性好 |

DeepSeek-R1-Zero 证明了纯 RL 训练推理能力的可行性，DeepSeek-R1 则在此基础上增加了 SFT 阶段，获得了更好的可用性。

## 四阶段训练流程

DeepSeek R1 的训练分为四个阶段：

### 阶段一：冷启动（Cold Start Phase）

**目标**：建立强大的可读性和响应质量基础

- 起点：DeepSeek-V3-Base 基础模型
- 方法：使用数千个来自 R1-Zero 的高质量验证样本进行**监督微调**
- 创新点：以少量高质量数据建立强基线的可读性和响应质量

### 阶段二：推理强化学习（Reasoning RL Phase）

**目标**：在数学、编程、科学、逻辑等领域培养核心推理能力

- 方法：基于规则的强化学习，奖励直接与**解答正确性**挂钩
- 关键特性：所有任务都是**可验证**的（如数学题可以用解题器验证答案）
- 创新点：直接优化，无需独立奖励模型，简化训练流程

### 阶段三：拒绝采样（Rejection Sampling Phase）

**目标**：质量控制，过滤低质量样本

- 方法：模型生成样本，由 DeepSeek-V3 作为质量评判者进行筛选
- 范围：不仅限于纯推理任务，覆盖更广泛的任务类型
- 筛选后的数据用于新一轮监督微调

### 阶段四：多样化强化学习（Diverse RL Phase）

**目标**：全面对齐，覆盖多种任务类型

- 方法：混合奖励策略
  - 确定性任务（如数学）→ 基于规则的奖励
  - 主观性任务（如写作）→ LLM 反馈作为奖励
- 目标：实现人类偏好对齐

## GRPO 算法详解

GRPO 是 DeepSeek R1 的核心算法创新，作者称其为模型微调的「突破」。与传统 RL 算法（如 PPO）相比，GRPO 能够「**直接优化偏好修正**」，提供更直接、更高效的对齐路径。

GRPO 由三个主要组件构成：

### 组件一：分组生成（Group Formation）

**核心思想**：像一个学生同时尝试多种解题思路

给定一个 prompt，模型不是只生成一个回答，而是生成**多个候选解答**（通常 4、8 或 16 个）。

以一道数学题为例，模型可能生成：
- 解法 A：逐步分解问题，先数总数再减去特定类别
- 解法 B：使用不同但同样有效的方法
- 解法 C：包含计算错误的错误解法
- 解法 D：思路正确但步骤不清晰

所有候选解法保留在同一个「组」中，供下一步比较。

### 组件二：偏好学习（Preference Learning）

**核心思想**：组内相对排名，不依赖绝对分数

对每个解法评估多个维度：
- 最终答案是否正确？
- 是否遵循了正确的格式（如 XML 标签）？
- 推理过程是否与给出的答案一致？

**优势计算公式**：

```
优势值 = (奖励值 - 组内平均奖励) / 组内奖励标准差
```

这种归一化就像「按曲线打分」：不看绝对分数，而是看相对于同组其他回答的表现。

### 组件三：策略优化（Policy Optimization）

**核心思想**：从经验中学习，同时保持稳定

优化过程遵循两个原则：
1. 鼓励模型生成更多「成功方案」，减少「失败方案」
2. 加入 **KL 散度惩罚**，防止模型一次性改变过大

相比传统方法更稳定的原因：
- 组内多个样本提供更稳定的梯度估计
- 组内归一化防止奖励尺度问题
- KL 惩罚确保模型在学习新技能的同时不遗忘已有知识

> **TIP**: GRPO 的关键创新点：
> 1. 可以从任意函数或模型中学习，不依赖独立奖励模型
> 2. 基于组的学习比逐对比较更稳定、更高效

In [2]:
# GRPO 算法伪代码（Python 形式，用于理解核心逻辑）

def grpo_algorithm_pseudocode():
    """
    GRPO 算法核心逻辑（伪代码形式）
    
    输入：
      - initial_policy: 待训练的初始模型
      - reward_function: 评估输出质量的奖励函数
      - training_prompts: 训练样本集合
      - group_size: 每个 prompt 生成的候选数（通常 4~16）
    """
    pass


# 用注释形式展示完整的 GRPO 伪代码逻辑
grpo_pseudocode = """
输入:
  initial_policy   = 初始策略模型（待训练）
  reward_function  = 评估输出的奖励函数（可以是任意函数）
  training_prompts = 训练样本集合
  group_size (G)   = 每个 prompt 的候选生成数（通常 4~16）

GRPO 算法主循环:
  for 每个训练迭代:
    a. 设置 reference_policy = current_policy（保存当前策略快照）
    
    b. for 每个 batch 中的 prompt:
       i.   用 initial_policy 生成 G 个不同的候选输出
            outputs = {o_1, o_2, ..., o_G}
       
       ii.  用 reward_function 计算每个输出的奖励
            rewards = {r_1, r_2, ..., r_G}
       
       iii. 组内奖励归一化（计算优势值）:
            advantage_i = (r_i - mean(rewards)) / std(rewards)
       
       iv.  更新策略，最大化以下目标函数:
            min(prob_ratio × advantage_i,
                clip(prob_ratio, 1-ε, 1+ε) × advantage_i)
            - kl_weight × KL(current_policy || reference_policy)
            
            其中 prob_ratio = π_θ(o_i|q) / π_θ_old(o_i|q)

输出: 优化后的策略模型
"""

print(grpo_pseudocode)


输入:
  initial_policy   = 初始策略模型（待训练）
  reward_function  = 评估输出的奖励函数（可以是任意函数）
  training_prompts = 训练样本集合
  group_size (G)   = 每个 prompt 的候选生成数（通常 4~16）

GRPO 算法主循环:
  for 每个训练迭代:
    a. 设置 reference_policy = current_policy（保存当前策略快照）

    b. for 每个 batch 中的 prompt:
       i.   用 initial_policy 生成 G 个不同的候选输出
            outputs = {o_1, o_2, ..., o_G}

       ii.  用 reward_function 计算每个输出的奖励
            rewards = {r_1, r_2, ..., r_G}

       iii. 组内奖励归一化（计算优势值）:
            advantage_i = (r_i - mean(rewards)) / std(rewards)

       iv.  更新策略，最大化以下目标函数:
            min(prob_ratio × advantage_i,
                clip(prob_ratio, 1-ε, 1+ε) × advantage_i)
            - kl_weight × KL(current_policy || reference_policy)

            其中 prob_ratio = π_θ(o_i|q) / π_θ_old(o_i|q)

输出: 优化后的策略模型



## 性能结果

DeepSeek R1 在多个领域达到了最先进的性能：

| 领域 | 评测基准 | DeepSeek R1 结果 |
|------|----------|------------------|
| **数学** | AIME 2024 | **79.8%** |
| **数学** | MATH-500 | **97.3%** |
| **编程** | Codeforces 评分 | **2029** |
| **编程** | LiveCodeBench | **65.9%** |
| **综合知识** | MMLU | **90.8%** |
| **综合知识** | GPQA Diamond | **71.5%** |
| **语言任务** | AlpacaEval 2.0 | **87.6% 胜率** |
| **语言任务** | FRAMES | **82.5%** |

### 模型蒸馏效果

DeepSeek R1 成功将能力蒸馏到更小的模型：

| 模型规模 | AIME 2024 成绩 |
|----------|----------------|
| 1.5B 参数 | 可验证的推理能力 |
| 7B 参数 | **55.5%** |
| 70B 参数 | 接近 o1-mini 性能（MATH-500: 94.5%） |

**API 成本**：仅需 $0.14/百万 input tokens，远低于竞品。

## GRPO 的局限性

尽管 GRPO 是重要进展，但仍有一些值得注意的局限：

| 局限性 | 说明 |
|--------|------|
| **生成成本高** | 每个 prompt 需生成 4~16 个候选，计算量是传统方法的数倍 |
| **Batch size 限制** | 需要将同一组的生成结果一起处理，限制了有效 batch size |
| **奖励函数设计难** | 设计不当的奖励可能导致「奖励黑客（Reward Hacking）」 |
| **组大小权衡** | 组太小多样性不足，组太大计算成本过高 |
| **KL 散度调参** | β 值需要仔细调整：过高学习太慢，过低可能不稳定 |

> **WARNING**: 奖励函数设计是 GRPO 成功的关键。设计不当的奖励函数可能导致模型「走捷径」优化奖励而不是真正学会任务。

## 本节小结

DeepSeek R1 论文代表了语言模型开发的重要里程碑：

1. **验证了纯 RL 可以发展推理能力**，挑战了必须依赖 SFT 的传统假设

2. **「顿悟时刻」自然涌现**，证明了学习而非记忆的真实推理能力

3. **GRPO 算法**通过组内比较和直接奖励优化，提供了高效的训练方案

4. **实用性强**：成功蒸馏到 1.5B~70B 多种规模，API 成本极低

### 思考练习

```
1. DeepSeek R1 论文的主要创新是什么？
    ① 算法层
        提出 GRPO（Group Relative Policy Optimization）
        用 group 相对比较 替代 value function（critic）
        降低 RLHF 训练成本
    ② 更重要的：训练范式
        👉 “RL-first reasoning emergence”不依赖大量SFT，仅通过RL让模型涌现推理能力
2. DeepSeek R1 训练过程的四个阶段分别是什么？
    阶段1：Cold Start（冷启动）
        少量高质量 CoT 数据
        目的是：
        给模型“起点”
        避免 RL 不稳定
    阶段2：Reasoning RL
        用 GRPO 做推理优化
        数学 / coding / logical tasks
    阶段3：Rejection Sampling + SFT
        从 RL 结果中筛高质量样本
        再做 SFT（很关键）
    阶段4：Alignment RL（多样化RL）
        对齐人类偏好（helpfulness / harmlessness）
3. 什么是 R1-Zero 训练中的「顿悟时刻」？
    模型在 RL 训练中，突然学会更有效的推理策略（如反思、分步推理、自我验证）
4. GRPO 的分组生成（Group Formation）是如何工作的？
    对同一 prompt 采样多个完整输出（sequence），这些输出构成一个 group，用于相对排序
5. DeepSeek-R1-Zero 和 DeepSeek-R1 的关键区别是什么？
    R1-Zero：纯RL（无SFT）→ 推理强但语言差
    R1：RL + SFT + alignment → 实用模型
```
---

**下一节**：我们将深入 GRPO 的数学原理，了解优势计算、目标函数、KL 散度等技术细节。

## 问答记录

本节汇总学习过程中产生的有价值问题及解答，供复习参考。

---

### Q1：KL 惩罚是什么？是什么缩写，怎么理解它的作用？

**KL 全称**：KL Divergence（KL 散度），KL 是两位统计学家 **Kullback** 和 **Leibler** 的名字缩写。

#### KL 散度的含义

KL 散度衡量**两个概率分布之间的差异程度**：

$$D_{KL}(P \| Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}$$

- 值为 0 → 两个分布完全相同  
- 值越大 → 两个分布差异越大

#### 在 RLHF / GRPO 中的作用

训练时存在两个模型：
- **Policy model**：正在被训练、权重持续更新的模型
- **Reference model**：训练前的原始模型，权重完全冻结

KL 惩罚体现在最终奖励的计算中：

$$r_{\text{final}} = r_{\text{reward}} - \beta \cdot D_{KL}(\pi_\theta \| \pi_{\text{ref}})$$

#### 为什么需要 KL 惩罚

不加 KL 惩罚会发生 **Reward Hacking**：

```
模型发现某种奇怪的输出能让奖励模型打高分
→ 疯狂朝那个方向优化
→ 输出变得荒谬，但 reward 很高
→ 模型彻底"跑偏"，丧失原有能力
```

KL 惩罚的作用是**拴住**训练中的模型，让它不能偏离原始模型太远：

```
KL 散度小 → 模型变化适度 → 惩罚小 → OK
KL 散度大 → 模型变化过大 → 惩罚大 → 被压制
```

#### 一句话理解

> **KL 惩罚是 RLHF 的"安全绳"**：允许模型变好，但不允许它为了刷高分而面目全非。

系数 $\beta$ 控制松紧——$\beta$ 越大，模型越保守；$\beta$ 越小，优化越激进。这也是 GRPO 局限性一节中提到"KL 散度调参需要仔细权衡"的原因。

---

### Q2：GRPO 得到组内奖励分数后，如何更新 policy？和普通梯度反向传播有什么异同？

#### 核心结论

**本质上还是梯度反向传播**，只是损失函数的构造方式不同。

#### GRPO 的损失函数

GRPO 的优化目标（最大化）：

$$\mathcal{L}_\text{GRPO} = \mathbb{E}\left[ \min\left( \frac{\pi_\theta(o_i|q)}{\pi_{\theta_\text{old}}(o_i|q)} \cdot A_i,\quad \text{clip}\left(\frac{\pi_\theta(o_i|q)}{\pi_{\theta_\text{old}}(o_i|q)}, 1-\varepsilon, 1+\varepsilon\right) \cdot A_i \right) \right] - \beta \cdot D_{KL}$$

三个关键组件：

**① 概率比（Probability Ratio）**

$$\text{ratio} = \frac{\pi_\theta(o_i|q)}{\pi_{\theta_\text{old}}(o_i|q)}$$

- 分子：当前模型对该输出的生成概率（参与梯度计算）
- 分母：采样时旧模型的概率（固定值，不参与梯度计算）
- ratio > 1 → 当前模型比旧模型更倾向于生成这个输出

**② 优势值（Advantage）**

$$A_i = \frac{r_i - \text{mean}(r_1, \ldots, r_G)}{\text{std}(r_1, \ldots, r_G)}$$

- $A_i > 0$：该输出比组内平均好 → 增大其概率
- $A_i < 0$：该输出比组内平均差 → 减小其概率

**③ Clip（截断）**

```
ratio 截断在 [1-ε, 1+ε] 范围内（通常 ε=0.2）
→ 防止单次更新幅度过大（取 clip 前后最小值，保守更新）
```

Loss 构造好后，后续流程和普通深度学习完全相同：

```
Loss → .backward() → 计算 ∂L/∂θ → optimizer.step() → 更新权重
```

#### 与监督学习（SFT）的对比

| 维度 | 监督学习（SFT） | GRPO |
|------|----------------|------|
| **Loss 来源** | 固定标签（正确答案） | 模型自采样输出 + 奖励 |
| **学习信号** | 直接告诉模型"输出什么" | 告诉模型"比同组平均好/差多少" |
| **梯度计算** | ∂CrossEntropy/∂θ | ∂(ratio × advantage)/∂θ |
| **反向传播** | ✅ 标准 backprop | ✅ 标准 backprop |
| **优化器** | Adam/SGD 等 | Adam/SGD 等（完全相同） |
| **目标** | 让输出接近标签分布 | 让高奖励输出的概率更高 |

#### 直觉理解

```
监督学习：老师给出标准答案，学生对照改正
    → Loss = 我的答案 vs 标准答案的差距

GRPO：学生同时写 8 份答卷，批改后比较
    → 让比平均分高的答案概率上升
    → 让比平均分低的答案概率下降
    → 没有"标准答案"，只有相对好坏
```

#### 关键洞察

GRPO 不需要知道"正确答案是什么"，只需要知道"哪个输出更好"——这正是它能用于数学验证、代码运行等场景的原因：这些场景只能判断对错，无法提供 token 级别的监督标签。